In [1]:
import os
import sys
import gc
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tabulate import tabulate
from huggingface_hub import hf_hub_download, login
from PIL import Image
import torchvision.transforms as T
import timm
from collections import defaultdict

# ==========================================
# 1. SETUP
# ==========================================
warnings.filterwarnings("ignore")
print("🔧 Setting up environment...")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

# ==========================================
# 2. THE "FORMAT-AGNOSTIC" ARCHITECTURE
# ==========================================
class NextSwinX(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.cnn_backbone = timm.create_model('convnext_base', pretrained=False, num_classes=0)
        self.swin_backbone = timm.create_model('swin_base_patch4_window7_224', pretrained=False, num_classes=0)
        
        self.cnn_backbone.head.norm = nn.LayerNorm(1024)
        
        self.proj_cnn = nn.Linear(1024, 512)
        self.proj_swin = nn.Linear(1024, 512)
        self.cross_attn = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)
        self.layer_norm = nn.LayerNorm(512)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, num_classes)
        )
        
    def _robust_flatten(self, x):
        if x.dim() == 2:
            return x
        if x.shape[1] == 1024:
            return x.mean(dim=tuple(range(2, x.dim())))
        elif x.shape[-1] == 1024:
            return x.mean(dim=tuple(range(1, x.dim()-1)))
        else:
            if x.shape[1] % 1024 == 0:
                return x.view(x.shape[0], 1024, -1).mean(dim=2)
            elif x.shape[-1] % 1024 == 0:
                return x.view(x.shape[0], -1, 1024).mean(dim=1)
            else:
                raise ValueError(f"Cannot determine feature dimension in shape {x.shape}")

    def forward(self, x):
        cnn_feat = self.cnn_backbone.forward_features(x)
        cnn_feat = self._robust_flatten(cnn_feat)
        cnn_feat = self.cnn_backbone.head.norm(cnn_feat)
        cnn_feat = self.proj_cnn(cnn_feat)
        
        swin_feat = self.swin_backbone.forward_features(x)
        swin_feat = self._robust_flatten(swin_feat)
        swin_feat = self.proj_swin(swin_feat)
        
        q = cnn_feat.unsqueeze(1)
        k = swin_feat.unsqueeze(1)
        v = k
        attn_out, _ = self.cross_attn(q, k, v)
        out = self.layer_norm(attn_out.squeeze(1))
        return self.classifier(out)

# ==========================================
# 3. ROBUST DATASET
# ==========================================
class RobustDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
        self.transform = T.Compose([
            T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(224),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        unique_labels = sorted(list(set(hf_dataset['label'])))
        if isinstance(unique_labels[0], str):
            self.label_map = {l: i for i, l in enumerate(unique_labels)}
        else:
            self.label_map = None

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        img = sample['image']
        label = sample['label']
        if isinstance(img, list):
            img = Image.fromarray(np.array(img, dtype=np.uint8).reshape(224, 224, 3))
        elif not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        if self.label_map:
            label = self.label_map[label]
        return self.transform(img), label

# ==========================================
# 4. CONFIGURATION (MISSING SEEDS ONLY)
# ==========================================
HF_TOKEN = "token"
REPO_ID = "srijan2715/reproductioncheckpoints"

print("🔐 Logging into Hugging Face...")
login(token=HF_TOKEN)

# UPDATED: Only the missing seeds!
models_config = {
    'NeXtSwin-X': {
        'class': NextSwinX,
        'paths': {
            'brisc': [
                "nextswinx_seed42_best(brisc).pth", 
                "nextswinx_seed43_best(brisc).pth"
            ], 
            'sartaj': [
                "proposed/nextswinx/sartaj/nextswinx_sartaj_seed44.pth"
            ]
        }
    }
}

datasets_config = {
    'brisc': 'srijan2715/brisc2025-paper-pipeline',
    'sartaj': 'srijan2715/sartaj-final-split'
}

# ==========================================
# 5. EVALUATION
# ==========================================
def evaluate(model_class, repo_filename, dataset_full_path, seed_num):
    local_path = None
    try:
        print(f"\n{'='*70}")
        print(f"🌱 Evaluating Seed {seed_num}...")
        print(f"{'='*70}")
        
        print("⬇️ Downloading...")
        local_path = hf_hub_download(repo_id=REPO_ID, filename=repo_filename, repo_type="model", token=HF_TOKEN)
        
        print("📂 Loading Dataset...")
        raw_test = load_dataset(dataset_full_path, split="test")
        test_ds = RobustDataset(raw_test)
        test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0)
        
        print("🧠 Loading Model...")
        model = model_class(num_classes=4).to(DEVICE)
        state_dict = torch.load(local_path, map_location=DEVICE)
        if 'model_state_dict' in state_dict: state_dict = state_dict['model_state_dict']
        
        print("🔒 Loading Weights...")
        model.load_state_dict(state_dict, strict=False)
        print("✅ Weights Loaded Successfully!")
            
        model.eval()
        all_preds, all_labels = [], []
        
        print("🚀 Running Inference...")
        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                _, preds = torch.max(out, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())
        
        acc = accuracy_score(all_labels, all_preds)
        prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
        rec = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
        f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
        
        print(f"\n✅ SUCCESS!")
        print(f"   🎯 Accuracy  : {acc:.4f}")
        print(f"   🎯 Precision : {prec:.4f}")
        print(f"   🎯 Recall    : {rec:.4f}")
        print(f"   🎯 F1-Score  : {f1:.4f}")
        
        return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1}, "Success"
        
    except Exception as e:
        print(f"❌ FAILED: {str(e)[:150]}")
        return None, str(e)
    finally:
        if 'model' in locals(): del model
        if local_path and os.path.exists(local_path): 
            try: os.remove(local_path)
            except: pass
        gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()

# ==========================================
# 6. RUN & AGGREGATE
# ==========================================
print("\n🚀 STARTING EVALUATION FOR MISSING SEEDS...")
results = []

for model_name, config in models_config.items():
    for ds_name, ds_path in datasets_config.items():
        if ds_name not in config['paths']: continue
        
        for repo_filename in config['paths'][ds_name]:
            # Extract seed number robustly
            if '42' in repo_filename: seed_num = 42
            elif '43' in repo_filename: seed_num = 43
            elif '44' in repo_filename: seed_num = 44
            else: seed_num = 0
            
            metrics, status = evaluate(config['class'], repo_filename, ds_path, seed_num)
            if metrics:
                results.append({
                    'Model': model_name, 
                    'Dataset': ds_name.capitalize(), 
                    'Seed': seed_num,
                    'Accuracy': metrics['acc'], 
                    'Precision': metrics['prec'],
                    'Recall': metrics['rec'], 
                    'F1-Score': metrics['f1']
                })

# ==========================================
# 7. FINAL OUTPUT
# ==========================================
print("\n" + "="*70)
print("🏆 NEW SEEDS RESULTS")
print("="*70)

if results:
    df_new = pd.DataFrame(results)
    print("\n" + tabulate(df_new, headers='keys', tablefmt='github', showindex=False))
    
    print("\n" + "="*70)
    print("📋 HOW TO UPDATE YOUR MASTER TABLE:")
    print("="*70)
    print("Combine these new results with your previous successful runs:")
    print("- BRISC: Seed 44 (0.8840) + these new BRISC seeds")
    print("- Sartaj: Seeds 42 (0.7792) & 43 (0.7030) + this new Sartaj seed")
    print("\nThen calculate the Mean ± Std Dev for all 3 seeds per dataset.")
else:
    print("No new seeds evaluated successfully.")

🔧 Setting up environment...
✅ Device: cuda
🔐 Logging into Hugging Face...

🚀 STARTING EVALUATION FOR MISSING SEEDS...

🌱 Evaluating Seed 42...
⬇️ Downloading...


nextswinx_seed42_best(brisc).pth: reconstructing file:   0%|          |  0.00B /  706MB            

nextswinx_seed42_best(brisc).pth: downloading bytes:           |  0.00B            

📂 Loading Dataset...


README.md:   0%|          | 0.00/540 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  907MB            

... (28 outputs left out)

Generating train split:   0%|          | 0/2439 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/431 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/394 [00:00<?, ? examples/s]

🧠 Loading Model...
🔒 Loading Weights...
✅ Weights Loaded Successfully!
🚀 Running Inference...

✅ SUCCESS!
   🎯 Accuracy  : 0.3655
   🎯 Precision : 0.2235
   🎯 Recall    : 0.3655
   🎯 F1-Score  : 0.2534

🏆 NEW SEEDS RESULTS

| Model      | Dataset   |   Seed |   Accuracy |   Precision |   Recall |   F1-Score |
|------------|-----------|--------|------------|-------------|----------|------------|
| NeXtSwin-X | Brisc     |     42 |   0.967    |    0.967253 | 0.967    |   0.967012 |
| NeXtSwin-X | Brisc     |     43 |   0.884    |    0.922016 | 0.884    |   0.882047 |
| NeXtSwin-X | Sartaj    |     44 |   0.365482 |    0.223524 | 0.365482 |   0.253375 |

📋 HOW TO UPDATE YOUR MASTER TABLE:
Combine these new results with your previous successful runs:
- BRISC: Seed 44 (0.8840) + these new BRISC seeds
- Sartaj: Seeds 42 (0.7792) & 43 (0.7030) + this new Sartaj seed

Then calculate the Mean ± Std Dev for all 3 seeds per dataset.
